# Часть 1. Загрузка и предобработка данных

В этом ноутбуке мы загружаем исходный датасет `NetflixShows`, изучаем его структуру, устраняем проблемы качества данных, такие как дубликаты, пропуски и избыточные признаки.

Результат этой части - это очищенный датасет `netflix_clean_1.csv`, который используется в последующих частях анализа.

#### Выполнил Ефимов Алексей


## Шаг 1. Загрузка и первичный осмотр данных

Читаем файл с явным указанием кодировки `utf-8-sig`, чтобы избежать артефакта BOM-метки
(`\xef\xbb\xbf`), который появляется при пересохранении файла из Excel в CSV.


In [191]:
import pandas as pd

data = pd.read_csv("NetflixShows.csv", encoding="utf-8-sig", sep=";")
data.head(3)

,title,rating,ratingLevel,ratingDescription,release year,user rating score,user rating size
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",80,2004,82.0,80
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",100,2006,NaN,82
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,90,2016,98.0,80


Представленный датасет содержит следующие признаки (колонки):

- `title` - название фильма, сериала
- `rating` - возрастной рейтинг
- `ratingLevel` - детализация, описание возрастного рейтинга
- `ratingDescription` - числовая кодировка рейтинга
- `release year` - дата выхода фильма или сериала
- `user rating score` - оценка контента пользователями
- `user rating size` - количество пользователей, которые поставили оценку

Признак `ratingDescription` - это просто числовая кодировка `rating`, которая у нас уже есть в текстовом виде. Он не несёт новой информации, поэтому удаляем его сразу.


In [192]:
del data["ratingDescription"]
data.head(3)

,title,rating,ratingLevel,release year,user rating score,user rating size
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,80


## Шаг 2. Дубликаты

При рассмотрении данных заметно, что одни и те же шоу встречаются несколько раз.
Часть из них - это сериалы. Также возможно данные были собраны парсингом или несколькими участниками, либо склейка файлов без проверки.


In [193]:
data[data['title'].str.lower() == 'black mirror']

,title,rating,ratingLevel,release year,user rating score,user rating size
17,Black Mirror,TV-MA,For mature audiences. May not be suitable for...,2016,80.0,80
74,Black Mirror,TV-MA,For mature audiences. May not be suitable for...,2016,80.0,80
480,Black Mirror,TV-MA,For mature audiences. May not be suitable for...,2016,80.0,80


In [194]:
data[data.duplicated()].sort_values("title").head(20)

,title,rating,ratingLevel,release year,user rating score,user rating size
396,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
241,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
347,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
141,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
295,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
497,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
189,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
201,30 Rock,TV-14,Parents strongly cautioned. May be unsuitable ...,2012,66.0,80
218,5 to 7,R,some sexual material,2014,NaN,82
265,90210,TV-14,Parents strongly cautioned. May be unsuitable ...,2013,62.0,80


Всего строк в файле 1000. Из них дублей - 500. Ровно половина датасета это дубликаты.


In [195]:
print(data.shape[0])
print(data.duplicated().sum())

1000
500


Применяем `drop_duplicates()`, потому что задублированы именно целые строки. После у нас остается 500 уникальных строк.

По каждому сериалу оставим одну строку, так как оги ничем не отличаются.


In [196]:
data_no_duplicates = data.drop_duplicates()
data_no_duplicates.shape[0]

500

Также необходимо сбросить индекс и переименовываем датафрейм, чтобы не путаться. Дальше работает с датасетом `data_clear`


In [197]:
data_no_duplicates.tail(5)

,title,rating,ratingLevel,release year,user rating score,user rating size
989,Russell Madness,PG,some rude humor and sports action,2015,NaN,82
993,Wiener Dog Internationals,G,General Audiences. Suitable for all ages.,2015,NaN,82
994,Pup Star,G,General Audiences. Suitable for all ages.,2016,NaN,82
997,Precious Puppies,TV-G,Suitable for all ages.,2003,NaN,82
998,Beary Tales,TV-G,Suitable for all ages.,2013,NaN,82


In [198]:
data_clear = data_no_duplicates.copy().reset_index(drop=True)
data_clear.tail(5)

,title,rating,ratingLevel,release year,user rating score,user rating size
495,Russell Madness,PG,some rude humor and sports action,2015,NaN,82
496,Wiener Dog Internationals,G,General Audiences. Suitable for all ages.,2015,NaN,82
497,Pup Star,G,General Audiences. Suitable for all ages.,2016,NaN,82
498,Precious Puppies,TV-G,Suitable for all ages.,2003,NaN,82
499,Beary Tales,TV-G,Suitable for all ages.,2013,NaN,82


## Шаг 3. Пропуски (NaN)


In [199]:
data_clear.isnull().sum()

title                  0
rating                 0
ratingLevel           33
release year           0
user rating score    244
user rating size       0
dtype: int64

Два столбца содержат пропуски:

- `ratingLevel` - текстовое описание рейтинга. Здесь 33 пропуска. Поскольку описание всегда одинаково для одного и того же `rating`, мы можем восстановить эти значения по уже имеющимся строкам.
- `user rating score` - оценка пользователей. Здесь уже 244 пропуска. Из 500 строк это очень много. Удалить или исключить из анализа столько наблюдений нельзя, поэтому в дальнейшем рассмотрим другие источники данных для получения рейтинга. Пока оставим без изменений.


### 3.1 Заполнение ratingLevel

Сделаем словарь `{rating: ratingLevel}` на основе строк, где значение `ratingLevel` уже есть. И выполним замену пропусков.


In [200]:
rating_to_level = (
    data_clear[data_clear["ratingLevel"].notna()]
    .groupby("rating")["ratingLevel"]
    .first()
    .to_dict()
)

data_clear["ratingLevel"] = data_clear["ratingLevel"].fillna(
    data_clear["rating"].map(rating_to_level)
)

data_clear["ratingLevel"].isnull().sum()

np.int64(0)

### 3.2 Формат названий колонок (признаков)

Приведем названия признаков к единому формату. Выбрали snake_case


In [201]:
data_clear = data_clear.rename(
    columns={
        "ratingLevel": "rating_level",
        "release year": "release_year",
        "user rating score": "rating_score",
        "user rating size": "rating_size",
    }
)
data_clear.columns.tolist()

['title',
 'rating',
 'rating_level',
 'release_year',
 'rating_score',
 'rating_size']

### 3.3 Финальная проверка


In [202]:
data_clear.isna().sum()

title             0
rating            0
rating_level      0
release_year      0
rating_score    244
rating_size       0
dtype: int64

## Шаг 4. Сохранение результата

Сохраняем итоговый датасет. Он будет использоваться далее для обогащения данными из внешних источников.


In [203]:
data_clear.to_csv("netflix_clean_1.csv", index=False)

## Итоги предобработки

Итоговый датасет: **500 строк, 6 признаков**.


In [204]:
data_clear.shape

(500, 6)